In [1]:
## Hannah's Work
## CNN Model

In [2]:
!pip install pyspedas
import pyspedas
from pytplot import get_data
import pandas as pd
import numpy as np
import xarray as xr
from datetime import timedelta
import os

# === Config ===
t1 = '2016-03-06/00:24:00'
t2 = '2016-03-06/00:34:00'
probes = [1]
buffer = 5  # minutes

# === Derived Setup ===
t1_ext = (pd.to_datetime(t1) - timedelta(minutes=buffer)).strftime('%Y-%m-%d/%H:%M:%S')
t2_ext = (pd.to_datetime(t2) + timedelta(minutes=buffer)).strftime('%Y-%m-%d/%H:%M:%S')
t1_dt = pd.to_datetime(t1).tz_localize('UTC')
t2_dt = pd.to_datetime(t2).tz_localize('UTC')
tag = t1.replace(":", "_").replace("/", "_") + "_to_" + t2.split("/")[1].replace(":", "_")
folder = f"data_{tag}"
os.makedirs(folder, exist_ok=True)

# === Helpers ===
def to_datetime_ns(arr): return pd.to_datetime(arr, unit='ns', utc=True)

def interp(original_time, original_data, target_time):
    original_sec = original_time.astype(int) / 1e9
    target_sec = target_time.astype(int) / 1e9
    if original_data.ndim == 1:
        return np.interp(target_sec, original_sec, original_data)
    return np.stack([np.interp(target_sec, original_sec, original_data[:, i]) for i in range(original_data.shape[1])], axis=1)

# === Data Variables to Load ===
mec_vars = ['r_gse', 'mlat', 'l_dipole', 'mlt', 'dst', 'kp']
var_labels = {
    'fgm_b_gse_srvy_l2_btot': 'Bt',
    'fgm_b_gse_srvy_l2': 'B_vec',
    'des_numberdensity_fast': 'Density',
    **{f'mec_{v}': v.upper() for v in mec_vars},
    'dsp_epsd_x': 'EPSD'
}

# === Loop ===
for probe in probes:
    # Load data
    pyspedas.mms.fgm(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.fpi(probe=probe, datatype=['des-moms'], center_measurement=True, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.mec(probe=probe, trange=[t1_ext, t2_ext], time_clip=True)
    pyspedas.mms.dsp(probe=probe, datatype=['epsd'], data_rate='fast', level='l2', trange=[t1_ext, t2_ext], time_clip=True)

    # Fetch and organize data
    raw_data = {k: get_data(f'mms{probe}_{k}', xarray=True) for k in var_labels}
    time_base = to_datetime_ns(raw_data['fgm_b_gse_srvy_l2_btot'].time.data)

    # Interpolation
    data_interp = {
        'Bt': raw_data['fgm_b_gse_srvy_l2_btot'].data,
        'B_vec': interp(to_datetime_ns(raw_data['fgm_b_gse_srvy_l2'].time.data), raw_data['fgm_b_gse_srvy_l2'].data, time_base),
        'Density': interp(to_datetime_ns(raw_data['des_numberdensity_fast'].time.data), raw_data['des_numberdensity_fast'].data, time_base)
    }

    for var in mec_vars:
        key = f'mec_{var}'
        data_interp[var.upper()] = interp(to_datetime_ns(raw_data[key].time.data), raw_data[key].data, time_base)

    # EPSD
    epsd_raw = raw_data['dsp_epsd_x']
    if epsd_raw is not None:
        epsd_time = to_datetime_ns(epsd_raw.time.data)
        freq = epsd_raw.v.data
        power = epsd_raw.data
        epsd_interp = np.stack([np.interp(time_base.astype(int)/1e9, epsd_time.astype(int)/1e9, power[:, i]) for i in range(len(freq))], axis=1)

        # Filter EPSD to 50–800 Hz
        freq_mask = (freq >= 50) & (freq <= 800)
        freq = freq[freq_mask]
        epsd_interp = epsd_interp[:, freq_mask]
    else:
        print(f"No EPSD data for MMS{probe}")
        freq = np.array([])
        epsd_interp = np.empty((len(time_base), 0))

    # Mask to user window
    mask = (time_base >= t1_dt) & (time_base <= t2_dt)
    time_final = time_base[mask]
    rel_time = (time_final - time_final[0]).total_seconds()

    # Create Dataset
    ds = xr.Dataset(
        {
            'Bt': ('time', data_interp['Bt'][mask]),
            'Bx': ('time', data_interp['B_vec'][mask, 0]),
            'By': ('time', data_interp['B_vec'][mask, 1]),
            'Bz': ('time', data_interp['B_vec'][mask, 2]),
            'Density': ('time', data_interp['Density'][mask]),
            'Rx': ('time', data_interp['R_GSE'][mask, 0]),
            'Ry': ('time', data_interp['R_GSE'][mask, 1]),
            'Rz': ('time', data_interp['R_GSE'][mask, 2]),
            'MLAT': ('time', data_interp['MLAT'][mask]),
            'L_DIPOLE': ('time', data_interp['L_DIPOLE'][mask]),
            'MLT': ('time', data_interp['MLT'][mask]),
            'DST': ('time', data_interp['DST'][mask]),
            'KP': ('time', data_interp['KP'][mask]),
            'EPSD': (['time', 'frequency'], epsd_interp[mask])
        },
        coords={
            'time': ('time', time_final),
            'frequency': ('frequency', freq),
            'RelativeTime': ('time', rel_time)
        },
        attrs={
            'description': f'MMS{probe} dataset with EPSD',
            'source': 'pySPEDAS and MMS L2'
        }
    )

    # Save
    out_file = f"{folder}/mms{probe}_multidimensional_{tag}.nc"
    ds.to_netcdf(out_file)
    print(f"Saved: {out_file}")

03-Jul-25 13:05:45: Loading files for group: probe: 1, drate: srvy, level: l2, datatype: , after sorting and filtering:
03-Jul-25 13:05:45: pydata/mms1/fgm/srvy/l2/2016/03/mms1_fgm_srvy_l2_20160306_v4.23.0.cdf
03-Jul-25 13:05:50: Loading files for group: probe: 1, drate: fast, level: l2, datatype: des-moms, after sorting and filtering:
03-Jul-25 13:05:50: pydata/mms1/fpi/fast/l2/des-moms/2016/03/mms1_fpi_fast_l2_des-moms_20160306000000_v3.4.0.cdf
03-Jul-25 13:05:51: Loading files for group: probe: 1, drate: srvy, level: l2, datatype: epht89q, after sorting and filtering:
03-Jul-25 13:05:51: pydata/mms1/mec/srvy/l2/epht89q/2016/03/mms1_mec_srvy_l2_epht89q_20160306_v2.2.1.cdf
03-Jul-25 13:05:52: Loading files for group: probe: 1, drate: fast, level: l2, datatype: epsd, after sorting and filtering:
03-Jul-25 13:05:52: pydata/mms1/dsp/fast/l2/epsd/2016/03/mms1_dsp_fast_l2_epsd_20160306_v0.6.0.cdf


Saved: data_2016-03-06_00_24_00_to_00_34_00/mms1_multidimensional_2016-03-06_00_24_00_to_00_34_00.nc


In [10]:
import os, numpy as np, pandas as pd, xarray as xr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import precision_score, recall_score, f1_score

# === Config ===
base_dir = "/Users/hannahw.owner/MMS Stuff/Machine learning"
data_folder = os.path.join(base_dir, "DATA_week")
sample_folder = os.path.join(base_dir, "All_samples")
test_file = os.path.join(data_folder, "mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc")
window_size, step_size = 160, 40
threshold = 0.675
batch_size = 64
epochs = 15
lr = 0.001

# === Model ===
class SimpleCNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.AdaptiveMaxPool1d(1)  # Changed from MaxPool to AvgPool
        )
        self.fc = nn.Linear(64, 1)  # No sigmoid

    def forward(self, x):
        x = self.conv(x)           # [batch, channels, 1]
        x = x.squeeze(-1)          # [batch, channels]
        return self.fc(x)          # [batch, 1]

# === Data Handling ===
def extract_features(ds):
    base = pd.DataFrame({v: ds[v].values for v in ["Bt", "Density", "MLT", "MLat", "Dst", "Kp"]})
    epsd = ds["EPSD"].values
    epsd_df = pd.DataFrame({f"EPSD_f{i}": epsd[:, i] for i in range(epsd.shape[1])})
    return pd.concat([base, epsd_df], axis=1)

def parse_filename_time(f):
    parts = f.replace(".nc", "").split('_')
    d = parts[2]
    s = pd.Timestamp(d) + pd.to_timedelta(f"{parts[3]}:{parts[4]}:{parts[5]}")
    e = pd.Timestamp(d) + pd.to_timedelta(f"{parts[7]}:{parts[8]}:{parts[9]}")
    return s, e

def label_window(t, intervals, duration=10):
    t_end = t + pd.Timedelta(seconds=duration)
    for istart, iend in intervals:
        if istart <= t_end and iend >= t:
            return 1
    return 0

class DuctingDataset(Dataset):
    def __init__(self, files, labeled_intervals):
        self.samples = []
        for file in files:
            ds = xr.open_dataset(file)
            times = pd.to_datetime(ds["time"].values)
            feats = extract_features(ds).values
            for i in range(0, len(feats) - window_size, step_size):
                window = feats[i:i + window_size]
                if np.isnan(window).any(): continue
                label = label_window(times[i], labeled_intervals)
                self.samples.append((window, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        x = torch.tensor(x.T, dtype=torch.float32)  # [features, time]
        y = torch.tensor(y, dtype=torch.float32)
        return x, y

# === Prepare Data ===
labeled_intervals = [parse_filename_time(f) for f in os.listdir(sample_folder) if f.endswith(".nc") and "2016-03-07" not in f]
train_files = [os.path.join(data_folder, f) for f in os.listdir(data_folder) if f.endswith(".nc") and "2016-03-07" not in f]

# Dataset and loader
dataset = DuctingDataset(train_files, labeled_intervals)
train_size = int(0.8 * len(dataset))
train_ds, val_ds = random_split(dataset, [train_size, len(dataset) - train_size])
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)

# === Model Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = dataset[0][0].shape[0]
model = SimpleCNN(n_features=input_dim).to(device)

# Compute class imbalance
pos_count = sum(label for _, label in dataset)
neg_count = len(dataset) - pos_count
pos_weight = torch.tensor([0.5 * (neg_count / (pos_count + 1e-5))]).to(device)  # Reduced weight

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# === Train ===
for epoch in range(epochs):
    model.train()
    epoch_loss = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb).squeeze()
        loss = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss.append(loss.item())
    print(f"Epoch {epoch+1}/{epochs}, Loss: {np.mean(epoch_loss):.4f}")

# === Evaluate ===
ds = xr.open_dataset(test_file)
features = extract_features(ds).values
times = pd.to_datetime(ds["time"].values)
ground_truth = [parse_filename_time(f) for f in os.listdir(sample_folder) if "2016-03-07" in f and f.endswith(".nc")]

predicted = []
model.eval()
with torch.no_grad():
    for i in range(0, len(features) - window_size, step_size):
        window = features[i:i + window_size]
        if np.isnan(window).any(): continue
        x = torch.tensor(window.T, dtype=torch.float32).unsqueeze(0).to(device)
        prob = torch.sigmoid(model(x)).item()
        if prob >= threshold:
            predicted.append((times[i], times[i + window_size - 1]))

# === Metrics ===
def overlap(a, b): return not (a[1] < b[0] or a[0] > b[1])
y_true, y_pred = [], []

for gt in ground_truth:
    match = any(overlap(gt, pred) for pred in predicted)
    y_true.append(1)
    y_pred.append(1 if match else 0)
for pred in predicted:
    if not any(overlap(pred, gt) for gt in ground_truth):
        y_true.append(0)
        y_pred.append(1)

print("\n--- SIMPLE CNN EVALUATION ---")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.2f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.2f}")
print(f"F1 Score:  {f1_score(y_true, y_pred, zero_division=0):.2f}")


Epoch 1/15, Loss: 0.4550
Epoch 2/15, Loss: 0.3758
Epoch 3/15, Loss: 0.3576
Epoch 4/15, Loss: 0.3447
Epoch 5/15, Loss: 0.3370
Epoch 6/15, Loss: 0.3303
Epoch 7/15, Loss: 0.3303
Epoch 8/15, Loss: 0.3209
Epoch 9/15, Loss: 0.3238
Epoch 10/15, Loss: 0.3194
Epoch 11/15, Loss: 0.3176
Epoch 12/15, Loss: 0.3184
Epoch 13/15, Loss: 0.3166
Epoch 14/15, Loss: 0.3124
Epoch 15/15, Loss: 0.3162

--- SIMPLE CNN EVALUATION ---
Precision: 0.10
Recall:    0.97
F1 Score:  0.18


In [12]:
import os, numpy as np, pandas as pd, xarray as xr
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from sklearn.metrics import precision_score, recall_score, f1_score

# === Config ===
base_dir = "/Users/hannahw.owner/MMS Stuff/Machine learning"
data_folder = os.path.join(base_dir, "DATA_week")
sample_folder = os.path.join(base_dir, "All_samples")
test_file = os.path.join(data_folder, "mms1_multidimensional_2016-03-07_00_00_06_to_23_59_59.nc")
window_size, step_size = 160, 40
threshold = 0.7
batch_size = 64
epochs = 15
lr = 0.001

# === Model ===
class SimpleCNN(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(n_features, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.AdaptiveMaxPool1d(1)
        )
        self.fc = nn.Linear(64, 1)

    def forward(self, x):
        x = self.conv(x)
        x = x.squeeze(-1)
        return self.fc(x)

# === Data Handling ===
def extract_features(ds):
    base = pd.DataFrame({v: ds[v].values for v in ["Bt", "Density", "MLT", "MLat", "Dst", "Kp"]})
    epsd = ds["EPSD"].values
    epsd_df = pd.DataFrame({f"EPSD_f{i}": epsd[:, i] for i in range(epsd.shape[1])})
    return pd.concat([base, epsd_df], axis=1)

def parse_filename_time(f):
    parts = f.replace(".nc", "").split('_')
    d = parts[2]
    s = pd.Timestamp(d) + pd.to_timedelta(f"{parts[3]}:{parts[4]}:{parts[5]}")
    e = pd.Timestamp(d) + pd.to_timedelta(f"{parts[7]}:{parts[8]}:{parts[9]}")
    return s, e

def label_window(t, intervals, duration=10):
    t_end = t + pd.Timedelta(seconds=duration)
    for istart, iend in intervals:
        if istart <= t_end and iend >= t:
            return 1
    return 0

class DuctingDataset(Dataset):
    def __init__(self, files, labeled_intervals):
        self.samples = []
        for file in files:
            ds = xr.open_dataset(file)
            times = pd.to_datetime(ds["time"].values)
            feats = extract_features(ds).values
            for i in range(0, len(feats) - window_size, step_size):
                window = feats[i:i + window_size]
                if np.isnan(window).any(): continue
                label = label_window(times[i], labeled_intervals)
                self.samples.append((window, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        x = torch.tensor(x.T, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        return x, y

# === Prepare Data ===
labeled_intervals = [parse_filename_time(f) for f in os.listdir(sample_folder) if f.endswith(".nc") and "2016-03-07" not in f]
train_files = [os.path.join(data_folder, f) for f in os.listdir(data_folder) if f.endswith(".nc") and "2016-03-07" not in f]

# Dataset and loader
dataset = DuctingDataset(train_files, labeled_intervals)
train_size = int(0.8 * len(dataset))
train_ds, val_ds = random_split(dataset, [train_size, len(dataset) - train_size])
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size)

# === Model Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_dim = dataset[0][0].shape[0]
model = SimpleCNN(n_features=input_dim).to(device)

# Compute class imbalance
pos_count = sum(label for _, label in dataset)
neg_count = len(dataset) - pos_count
pos_weight = torch.tensor([0.5 * (neg_count / (pos_count + 1e-5))]).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# === Train ===
for epoch in range(epochs):
    model.train()
    epoch_loss = []
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb).squeeze()
        loss = criterion(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss.append(loss.item())
    print(f"Epoch {epoch+1}/{epochs}, Loss: {np.mean(epoch_loss):.4f}")

# === Evaluate ===
ds = xr.open_dataset(test_file)
features = extract_features(ds).values
times = pd.to_datetime(ds["time"].values)
ground_truth = [parse_filename_time(f) for f in os.listdir(sample_folder) if "2016-03-07" in f and f.endswith(".nc")]

predicted = []
model.eval()
with torch.no_grad():
    for i in range(0, len(features) - window_size, step_size):
        window = features[i:i + window_size]
        if np.isnan(window).any(): continue
        x = torch.tensor(window.T, dtype=torch.float32).unsqueeze(0).to(device)
        prob = torch.sigmoid(model(x)).item()
        if prob >= threshold:
            predicted.append((times[i], times[i + window_size - 1]))

# === Metrics ===
def overlap(a, b): return not (a[1] < b[0] or a[0] > b[1])
y_true, y_pred = [], []

for gt in ground_truth:
    match = any(overlap(gt, pred) for pred in predicted)
    y_true.append(1)
    y_pred.append(1 if match else 0)
for pred in predicted:
    if not any(overlap(pred, gt) for gt in ground_truth):
        y_true.append(0)
        y_pred.append(1)

print("\n--- SIMPLE CNN EVALUATION ---")
print(f"Precision: {precision_score(y_true, y_pred, zero_division=0):.2f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.2f}")
print(f"F1 Score:  {f1_score(y_true, y_pred, zero_division=0):.2f}")


Epoch 1/15, Loss: 0.3844
Epoch 2/15, Loss: 0.3501
Epoch 3/15, Loss: 0.3279
Epoch 4/15, Loss: 0.3212
Epoch 5/15, Loss: 0.3152
Epoch 6/15, Loss: 0.3086
Epoch 7/15, Loss: 0.3033
Epoch 8/15, Loss: 0.2983
Epoch 9/15, Loss: 0.2940
Epoch 10/15, Loss: 0.2931
Epoch 11/15, Loss: 0.2903
Epoch 12/15, Loss: 0.2872
Epoch 13/15, Loss: 0.2872
Epoch 14/15, Loss: 0.2880
Epoch 15/15, Loss: 0.2864

--- SIMPLE CNN EVALUATION ---
Precision: 0.14
Recall:    0.87
F1 Score:  0.25
